# Stock Data Downloader (20 Years, Daily)

This notebook downloads ~20 years of daily OHLCV data for a large list of tickers
and writes it into a SQLite database plus a `.sql` dump.

Prerequisites:
- `pip install -r requirements.txt`
- A CSV file with at least 1000 tickers (one per row).

CSV format example (any header is fine):
```
ticker
AAPL
MSFT
...
```


In [ ]:
import time
import sqlite3
from pathlib import Path
import pandas as pd
from yahoo_finance import Share


In [ ]:
# Config
TICKERS_CSV = Path('dataset/selected_1000_tickers.csv')  # update if needed
DB_PATH = Path('dataset/stock_prices_20y.db')
SQL_DUMP_PATH = Path('dataset/stock_prices_20y.sql')

# Download settings
BATCH_PAUSE_SEC = 1.0   # pause between tickers to reduce rate limiting
MAX_RETRIES = 3
BACKOFF_SEC = 2.0

# Date range: last 20 years (approx, based on today)
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=20)

start_str = start_date.strftime('%Y-%m-%d')
end_str = end_date.strftime('%Y-%m-%d')

print('Date range:', start_str, 'to', end_str)


In [ ]:
# Load tickers
if not TICKERS_CSV.exists():
    raise FileNotFoundError(f'Missing tickers file: {TICKERS_CSV}')

tickers_df = pd.read_csv(TICKERS_CSV)
if 'ticker' in tickers_df.columns:
    tickers = tickers_df['ticker'].astype(str).str.strip().tolist()
elif tickers_df.shape[1] == 1:
    tickers = tickers_df.iloc[:, 0].astype(str).str.strip().tolist()
else:
    # Try common column names
    for col in ['symbol', 'Ticker', 'Symbol']:
        if col in tickers_df.columns:
            tickers = tickers_df[col].astype(str).str.strip().tolist()
            break
    else:
        raise ValueError('Ticker column not found. Use a single-column CSV or name the column ticker/symbol.')

tickers = [t for t in tickers if t]
print('Total tickers:', len(tickers))
if len(tickers) < 1000:
    raise ValueError('Need at least 1000 tickers in the CSV.')



In [ ]:
# Initialize database
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
conn = sqlite3.connect(DB_PATH)

conn.execute(
    '''
    CREATE TABLE IF NOT EXISTS prices (
        ticker TEXT NOT NULL,
        date TEXT NOT NULL,
        open REAL,
        high REAL,
        low REAL,
        close REAL,
        adj_close REAL,
        volume INTEGER
    )
    '''
)

conn.execute(
    'CREATE INDEX IF NOT EXISTS idx_prices_ticker_date ON prices(ticker, date)'
)
conn.commit()

def already_loaded_tickers(connection):
    rows = connection.execute('SELECT DISTINCT ticker FROM prices').fetchall()
    return set(r[0] for r in rows)

loaded = already_loaded_tickers(conn)
print('Tickers already in DB:', len(loaded))


In [ ]:
def fetch_history(ticker):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            share = Share(ticker)
            data = share.get_historical(start_str, end_str)
            if not data:
                return pd.DataFrame()
            df = pd.DataFrame(data)
            if 'Date' not in df.columns:
                return pd.DataFrame()
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
            df = df.dropna(subset=['Date'])
            df = df.sort_values('Date')
            df['ticker'] = ticker

            # Normalize column names
            rename_map = {
                'Date': 'date',
                'Open': 'open',
                'High': 'high',
                'Low': 'low',
                'Close': 'close',
                'Adj_Close': 'adj_close',
                'Adj Close': 'adj_close',
                'Volume': 'volume',
            }
            df = df.rename(columns=rename_map)

            for col in ['open','high','low','close','adj_close','volume']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            # Keep only known columns
            cols = ['ticker','date','open','high','low','close','adj_close','volume']
            return df[cols]
        except Exception as exc:
            if attempt == MAX_RETRIES:
                print(f'Failed: {ticker} after {MAX_RETRIES} attempts. Error: {exc}')
                return pd.DataFrame()
            sleep_s = BACKOFF_SEC * attempt
            print(f'Retry {attempt}/{MAX_RETRIES} for {ticker} in {sleep_s:.1f}s...')
            time.sleep(sleep_s)

def save_to_db(connection, df):
    if df.empty:
        return 0
    df = df.copy()
    df['date'] = df['date'].dt.strftime('%Y-%m-%d')
    df.to_sql('prices', connection, if_exists='append', index=False)
    connection.commit()
    return len(df)


In [ ]:
# Main download loop
processed = 0
inserted = 0

for ticker in tickers:
    if ticker in loaded:
        continue
    df = fetch_history(ticker)
    n = save_to_db(conn, df)
    processed += 1
    inserted += n
    if processed % 10 == 0:
        print(f'Processed {processed} tickers, inserted {inserted} rows')
    time.sleep(BATCH_PAUSE_SEC)

print('Done. Total inserted rows:', inserted)


In [ ]:
# Export SQL dump (can be large)
with open(SQL_DUMP_PATH, 'w', encoding='utf-8') as f:
    for line in conn.iterdump():
        f.write(f'{line}\n')

print('SQL dump written to', SQL_DUMP_PATH)
